## S. pombe
H Nakaoka, Y Wakamoto, Aging, mortality, and the fast growth trade-off of Schizosaccharomyces pombe. PLoS Biol. 15, e2001109 (2017).

In [1]:
# include packages
using LinearAlgebra
using DifferentialEquations, Plots
using BenchmarkTools
using HypothesisTests
default(grid=false,legend=false, size=(400,320), dpi=300, tickfontsize=9, lw=2, lc=:gray, labelfontsize=9, tickdir=:out, fontfamily="Arial", legendfontsize=7, markercolor=:indianred, thickness_scaling=2)

mydir = "../helper/"
include(mydir * "base.jl")
include(mydir * "optimization.jl")
include(mydir * "misc.jl")
include(mydir * "myplot.jl")

plot_output (generic function with 3 methods)

In [12]:
# read in data
data = readdlm("../data/doi_10/Results_YE_30degree.txt", ' ');
area_temp = [];
area_obs = [];
tobs = [];
ttemp = [];
open("../data/doi_10/Results_YE_30degree.txt", "r") do file
    for line in eachline(file)
        parts = split(line, '\t')
        push!(area_temp, parts[3])
        push!(ttemp, parts[7])
        if parts[8] == "0"
            push!(area_obs, copy(area_temp))
            empty!(area_temp)
            push!(tobs, copy(ttemp))
            empty!(ttemp)
        end
    end
end

In [13]:
for i in eachindex(area_obs)
    area_obs[i] = parse.(Float64, area_obs[i][2:end]);
    tobs[i] = parse.(Float64, tobs[i][2:end]);
end

In [14]:
area_obs_ = convert(Vector{Vector{Float64}}, area_obs);
tobs_ = convert(Vector{Vector{Float64}}, tobs);


In [15]:
area_, t_, isdiv_, dts_, llast_, llast2_ = annotate_exp_traj4(area_obs_, tobs_, cutoff = 100.0)

([[253.0, 268.0, 261.0, 278.0, 267.0, 270.0, 287.0, 304.0, 298.0, 306.0  …  338.0, 346.0, 350.0, 352.0, 347.0, 338.0, 332.0, 337.0, 342.0, 348.0], [303.0, 310.0, 312.0, 324.0, 320.0, 327.0, 316.0, 318.0, 314.0, 315.0  …  374.0, 357.0, 368.0, 373.0, 377.0, 383.0, 386.0, 378.0, 383.0, 402.0], [146.0, 162.0, 157.0, 180.0, 177.0, 166.0, 177.0, 181.0, 182.0, 198.0  …  325.0, 317.0, 324.0, 316.0, 321.0, 314.0, 333.0, 335.0, 339.0, 371.0], [173.0, 173.0, 188.0, 188.0, 192.0, 193.0, 204.0, 214.0, 207.0, 212.0  …  327.0, 331.0, 334.0, 324.0, 321.0, 321.0, 337.0, 336.0, 342.0, 361.0], [230.0, 225.0, 236.0, 236.0, 239.0, 250.0, 258.0, 259.0, 267.0, 271.0  …  328.0, 332.0, 336.0, 334.0, 344.0, 340.0, 345.0, 345.0, 363.0, 391.0], [182.0, 190.0, 194.0, 193.0, 192.0, 189.0, 204.0, 194.0, 202.0, 209.0  …  309.0, 310.0, 311.0, 311.0, 314.0, 313.0, 310.0, 313.0, 317.0, 315.0], [304.0, 324.0, 310.0, 329.0, 328.0, 144.0, 147.0, 164.0, 175.0, 170.0  …  340.0, 338.0, 345.0, 343.0, 339.0, 357.0, 349.0, 341.0

In [16]:
function truncation(area, tobs, isdiv, upperbound, lowerbound)
    exclude = findall(area.>upperbound .|| area.<lowerbound);
    (isempty(exclude)) && (return area, tobs)
    if exclude[1] == 1
        return truncation(area[2:end], tobs[2:end], isdiv[2:end], upperbound, lowerbound)
    end
    first = 1;
    # last = exclude[1] - 1;
    last = findlast(isdiv[1:exclude[1] - 1]);
    (isnothing(last)) && (last = exclude[1])
    truncated = Vector{Float64}[];
    truncated_time = Vector{Float64}[];
    truncated = vcat(truncated, [area[first:last]]);
    truncated_time = vcat(truncated_time, [tobs[first:last]]);
    return truncated, truncated_time
end

truncation (generic function with 1 method)

In [17]:
areaobs__=[]
tobs__ = []
for i in eachindex(area_)
    aa, tt = truncation(area_[i], t_[i], isdiv_[i], 500, 100)
    if isa(aa, Vector{Vector{Float64}})
        push!(areaobs__, aa[1]);
        push!(tobs__, tt[1]);
    else
        push!(areaobs__, aa);
        push!(tobs__, tt);
    end
end
areaobs__ = convert(Vector{Vector{Float64}}, areaobs__);
tobs__ = convert(Vector{Vector{Float64}}, tobs__);

In [18]:
tobs__ ./= 20.0;

In [19]:
areaobs__, tobs__, isdiv_, dtobs_, last_, last2_ = annotate_exp_traj4(areaobs__, tobs__, cutoff = 100.0);

In [20]:
areaobs__ = areaobs__.*0.0935;
last_ = last_.*0.0935;
last2_ = last2_.*0.0935;

In [21]:
sum(sum.(isdiv_))

56154

### trajectory

In [23]:
p_traj = plot(; size=(750,300),xlim=(0,50), ylim=(0,50), xtick=collect(0:25:100), ytick=collect(0:25:50), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent)
linecolor_ = palette(:default, length(area_) )
for i =1:50
    plot!(p_traj, tobs__[i], areaobs__[i], lc=:snow2, lw=0.2)
end
pj = 5
plot!(p_traj, tobs__[pj], areaobs__[pj], lw = 0.4)
scatter!(p_traj, tobs__[pj][isdiv_[pj]], areaobs__[pj][isdiv_[pj]], markercolor=:gray, ms=1,markerstrokewidth=0.2)
p_traj
savefig("../figure/fig3/traj_yeast.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/traj_yeast.pdf"

### correlation

In [25]:
using StatsBase
hedges = collect(25:1.0:45);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h2_obs = fit(Histogram, (vcat( [areaobs__[i][isdiv_[i]][1:end-1] for i in eachindex(areaobs__) ]...), vcat( [areaobs__[i][isdiv_[i]][2:end] for i in eachindex(areaobs__) ]...) ), (hedges,hedges));

h2_obs_normalize = h2_obs.weights / sum(h2_obs.weights) / dhedges^2

svd_h2_obs = svd(h2_obs_normalize);

ll=@layout [a b]
p_h2_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(hedges[1],hedges[end]), ylim=(hedges[1],hedges[end]), 
                xtick=collect(15:10:55), ytick=collect(15:10:55) )#, title="P(i,i-1)"
heatmap!( p_h2_obs, hbins, hbins, h2_obs_normalize, clim=(0,2e-2),c = :Blues_9,
colorbar_ticks=()) #,colorbar=true

plot(p_h2_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/fig3/yeast_correlation.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_correlation.png"

In [28]:
p_S_obs = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:9), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_obs, svd_h2_obs.S ./ svd_h2_obs.S[1], markercolor=:gray, ms=3, markershape=:hline,msw=4)
savefig("../figure/fig3/yeast_svd.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_svd.pdf"

In [30]:
# mutual information
P_xy = h2_obs.weights/sum(h2_obs.weights)
P_y = sum(P_xy, dims=1)
P_x = sum(P_xy, dims=2)
is_nan = .!(isnan.(P_xy .* log.(P_xy./(P_x * P_y))))
mi = P_xy .* log.(P_xy./(P_x * P_y))
sum(mi[is_nan])

0.045236116201459425

### input data distribution

In [31]:
using StatsBase
hedges = collect(25:1.0:45);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
hh_obs = fit(Histogram, (vcat( [areaobs__[i] for i in eachindex(areaobs__) ]...), vcat( [last_[i] for i in eachindex(last_) ]...) ), (hedges,hedges));

hh_obs_normalize = hh_obs.weights / sum(hh_obs.weights) / dhedges^2

ll=@layout [a b]
p_hh_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(25,45), ylim=(25,45), xtick=collect(25:10:45), ytick=collect(25:10:45))#, title="P(i,i-1)" 
heatmap!( p_hh_obs, hbins, hbins, hh_obs_normalize, clim=(0,0.02),c = :Blues_9,
colorbar_ticks=(),colorbar=false) 

plot(p_hh_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=5Plots.mm, background_color=:transparent)
savefig("../figure/fig3/yeast_datadist.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_datadist.png"

### inference

In [497]:
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
n_func1 = [3, 3]
l01 = [30.0, 30.0]
Θ1, Σ1 = mgs_gen_bases( [vcat(areaobs__...), vcat(last_...)], n_func1, bf1, l01);
Θdiv1 = Θ1[vcat(isdiv_...),:];
Θdiv1 = sum(Θdiv1, dims=1)[:];

history1 = sparse_optimization_sweep(sparse_optimization_fg, Θ1, Θdiv1, vcat(dtobs_...); init_cutoff=0.0, coeffs0=1e-4, EPSILON=1E-7, MAX_ITERS=20);

===== start of epoch 1 =====


itr=1: 14315.939487654425


itr=2: 14253.99973446479


itr=3: 14255.309937788466


itr=4: 14256.19540364424


itr=5: 14257.546556701296


... (167 outputs left out)

itr=1: 107319.66304440133


itr=2: 107319.81872451639


itr=3: 107319.81871929075


===== end of epoch 10 =====
error = 1.0000000000340654, nterms = 0, cutoff = 1.8422433158187879


In [498]:
save_inference_results([vcat(tobs__...) vcat(areaobs__...) vcat(last_...)], vcat(isdiv_...), vcat(dtobs_...), 
    [ [@sprintf "%s" typeof(bf1[i]) for i in 1:length(bf1)] n_func1 l01 ], history1; 
    data_type = "exp_yeast", learn_type = "learn_memory1", output_dir = "./output/", 
    use_uid=true, save_traj=true
)

### plotting

In [32]:
filepath1 = "./output/exp_yeast_learn_memory1/23-11-27-T1940/"

traj = readdlm(filepath1 * "traj");
tobs = traj[:,1];
Ls1 = [traj[:,j] for j in 2:3];

dtobs = readdlm(filepath1 * "dts")[:];
isdiv = convert( BitVector, readdlm(filepath1 * "isdiv")[:] );

In [34]:
# rebuild inference basis function library

function rebuild_library_params(filepath)
    library_sum = readdlm(filepath * "library");
    bf_string = library_sum[:, 1];
    @show bf_string
    n_func = convert(Vector{Int}, library_sum[:,2]);
    l0 = convert(Vector{Float64}, library_sum[:,3]);
    return n_func, l0
end

n_func1, l01 = rebuild_library_params(filepath1);
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()];
_, Σ1 = mgs_gen_bases(Ls1, n_func1, bf1, l01);

bf_string = Any["sChebyshevT{Float64}", "sChebyshevT{Float64}"]


In [35]:
# read in inference result
function rebuild_history(filepath)

    history_coeffs = readdlm(filepath * "history_coeffs");
    cache_summary  = readdlm(filepath * "history_summary");
    keynames = convert(Vector{String}, cache_summary[1,:] );
    history_metric = cache_summary[2:end,:];
    history = Dict(zip(Symbol.(keynames), [history_metric[:,i] for i = 1:length(keynames)]));

    return history, history_coeffs

end

history1, history_coeffs1 = rebuild_history(filepath1);

In [37]:
# find the maximum BIC score
_, imax1 = findmax( -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs) )


BIC = -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs)
plot(history1[:nterms], BIC, lw=2, lc=:dodgerblue4,
    xtick=[0,10,20],ytick=[-1,-0.5,0],ylim=(-1.1,0), size=(330,300), bottommargin=-6Plots.mm, leftmargin=-10Plots.mm,
    background_color=:transparent)
scatter!([history1[:nterms][imax1]], [BIC[imax1]], ms=2.5, mc=:dodgerblue4, shape=:rect)
savefig("../figure/fig3/yeast_BIC.pdf")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_BIC.pdf"

In [42]:
# calculate the inferred λ
ll = collect(15.0:1:45.0);
ll_ = collect(15.0:1:45.0);
logλ_learn1 = mgs_rec_bases([ll, ll_], Σ1, history_coeffs1[:, imax1], bf1, l01);

pl1 = plot(;size=(300,300), bottommargin=-6Plots.mm, leftmargin=-3Plots.mm, background_color=:transparent);
heatmap!(pl1, ll, ll_, exp.(logλ_learn1'), clim=(0,2),
xlim=(25,45), ylim=(25,45), aspect_ratio=1, framestyle=:box, 
c=:lapaz, xtick=[25,35,45],ytick=[25,35,45])
savefig("../figure/fig3/yeast_memory1.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_memory1.png"

In [44]:
heatmap(reshape(history_coeffs1[:, imax1],5,5), c =:broc, ticks=false, axis=true,aspect_ratio =1, clim=(-1,1),
xlim=(0.5,5.5), ylim=(0.5,5.5),framestyle=:box,size=(300,300), background_color=:transparent, yflip = true
)
savefig("../figure/fig3/yeast_coeffs_matrix.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_coeffs_matrix.png"

### Growth function g inference

In [45]:
# interpolation to smooth the growth function g
function interpolation(t, L, deg)
    A = ones(length(L), deg);
    for i = 2:deg
        A[:,i] = t.^(i-1);
    end
    coeff = A\L;
    D = zeros(length(L), deg);
    for i = 2:deg
        D[:,i] = (i-1) * t.^(i-2);
    end
    return coeff, D * coeff, A * coeff
end

interpolation (generic function with 1 method)

In [48]:
bf_g = Polynomial()
Lt_g = []
Diff_g = []
for Li in eachindex(areaobs__)
    pos = findall(isdiv_[Li]);
    for pi = 1:length(pos)-1
        _,diff, Lt = interpolation(tobs__[Li][pos[pi]+1:pos[pi+1]], areaobs__[Li][pos[pi]+1:pos[pi+1]], 5);
        push!(Lt_g, Lt);
        push!(Diff_g, diff);
    end
end

# linear regression to learn growth rate function g
Lt_g = convert(Vector{Vector{Float64}}, Lt_g);
Diff_g = convert(Vector{Vector{Float64}}, Diff_g);
Lt_g = vcat( [ Lt_g[i]  for i in eachindex(Lt_g)]... );
Diff_g = vcat( [Diff_g[i] for i in eachindex(Diff_g) ]... );
Theta_g = gen_bases( Lt_g,  2);
coeffs_g = Theta_g[:,[1,2]] \ Diff_g

2-element Vector{Float64}:
  8.264026453003089
 -0.025868087417053688

In [50]:
histogram2d(Lt_g, Diff_g, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-10Plots.mm,
xlim = (5,45), ylims = (-25,45), xtick =[10, 25, 40], ytick=[-20, 0, 20, 40], bins=(5:2.5:45, -25:5:45), c=cgrad(:tempo, rev=false), clim=(0,0.01)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,
plot!([5,46], coeffs_g[1] .+ coeffs_g[2]*[5,46], lw=1.0, lc=:black, ls=:dash)
savefig("../figure/fig3/yeast_infered_g.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_infered_g.png"

### Cut size function h inference

In [54]:
# linear regression to learn cut size function h
bf_h = Polynomial()
Y_h = vcat( [areaobs__[i][isdiv_[i]][1:end-1] .- areaobs__[i][findall(isdiv_[i][1:end-1]) .+  1]  for i in eachindex(areaobs__)]... );
Theta_h = gen_bases( vcat( [areaobs__[i][isdiv_[i]][1:end-1] for i in eachindex(areaobs__) ]... ), 1);

coeffs_h = Theta_h\ Y_h
coeffs_h[abs.(coeffs_h) .< 0.02] .= 0.0

0-element view(::Vector{Float64}, Int64[]) with eltype Float64

In [58]:
histogram2d( vcat( [areaobs__[i][isdiv_[i]][1:end-1] for i in eachindex(areaobs__) ]... ), Y_h, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm,
xlim = (20,50), ylim = (12,25), xtick =[30, 40, 50], ytick=[10, 15, 20,25], bins=(20:1:50, 10:1:25), c=cgrad(:tempo, rev=false), clim=(0, 0.05)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,)
plot!([12, 50], coeffs_h[1] .+ coeffs_h[2] * [12, 50].^1, lw=1.0, lc=:black, ls=:dash)
savefig("../figure/fig3/yeast_infered_h.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_infered_h.png"

### Resimulation

In [59]:
function f(du, u, p, t)
    bf_f = p[1]
    coeffs_f = p[2]
    du[1] = 0.0
    for n = 1:length(coeffs_f)
        du[1] += coeffs_f[n]*bf_f(n-1, u[1])
    end
end

function rate_memory(u, p, t)
    Σ1 = p[5]
    coeffs = p[6]
    bf1 = p[7]
    l01 = p[8]
    ln_rate = mgs_rec_bases([[u[1]], [u[2]]], Σ1, coeffs, bf1, l01);
    return exp(ln_rate[1])
end

function affect!(integrator)
    integrator.u[2] = integrator.u[1]
    bf_g = integrator.p[3]
    coeffs_g = integrator.p[4]
    u_old = integrator.u[1]
    for n = 1:length(coeffs_g)
        integrator.u[1] -= coeffs_g[n] * bf_g(n-1, u_old)
    end
end

affect! (generic function with 1 method)

In [60]:
# resimulate data and save or read in your resimulated data
# resimulate data based on your inferred parameters
Lresult = []
tresult = []
for i in eachindex(areaobs__)
    vrj = VariableRateJump(rate_memory, affect!)
    oprob = ODEProblem(f, [areaobs__[i][1];2 * areaobs__[i][1]], (0.0,maximum(tobs__[i])), (bf_g, coeffs_g, bf_h, coeffs_h, Σ1, history_coeffs1[:, imax1], bf1, l01))
    jprob = JumpProblem(oprob, Direct(), vrj)
    sol = solve(jprob,Tsit5(),dtmax=0.01);
    L_r = Array(sol')[:,1];
    t_r = sol.t;
    push!(Lresult, L_r);
    push!(tresult, t_r);
end

Lresult = convert(Vector{Vector{Float64}}, Lresult);
tresult = convert(Vector{Vector{Float64}}, tresult);

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

writedlm("../figure/fig3/yeast_resim_data.txt", [Lresult, tresult], '\t')

In [64]:
# read in resimulated data

using JSON
resim_data = readdlm("../figure/fig3/yeast_resim_data.txt", '\t')

Lresult = [Float64.(JSON.parse(resim_data[1, i])) for i in 1:size(resim_data,2)];
tresult = [Float64.(JSON.parse(resim_data[2, i])) for i in 1:size(resim_data,2)];

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

In [ ]:
p_traj = plot(; size=(750,300),xlim=(0,50), ylim=(0,50), xtick=collect(0:25:100), ytick=collect(0:25:50), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent)
linecolor_ = palette(:default, length(area_) )
for i =1:50
    plot!(p_traj, tresult[i], Lresult[i], lc=:snow2, lw=0.2)
end
plot!(p_traj, tresult[3], Lresult[3], lc=:seagreen,lw=0.4)
scatter!(p_traj, tresult[3][isdiv_result[3]], Lresult[3][isdiv_result[3]], markercolor=:seagreen, ms=1,markerstrokewidth=0.2)
p_traj
savefig("../figure/fig3/traj_yeast_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/traj_yeast_resim.pdf"

In [67]:
using StatsBase
hedges = collect(25:1:45);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h2_result = fit(Histogram, (vcat( [Lresult[i][isdiv_result[i]][1:end-1] for i in eachindex(Lresult) ]...), vcat( [Lresult[i][isdiv_result[i]][2:end] for i in eachindex(Lresult) ]...) ), (hedges,hedges));

h2_result_normalize = h2_result.weights / sum(h2_result.weights) / dhedges^2

svd_h2_result = svd(h2_result_normalize);

ll=@layout [a b]
p_h2_result = plot(; aspect_ratio=1, framestyle=:box, xlim=(25,45), ylim=(25,45), xtick=collect(15:10:55), ytick=collect(15:10:55))#, title="P(i,i-1)" 
heatmap!( p_h2_result, hbins, hbins, h2_result_normalize,c = :Blues_9, clim=(0,2e-2),
colorbar_ticks=()) 

plot(p_h2_result, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/fig3/yeast_correlation_resim.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_correlation_resim.png"

In [69]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h2_obs.S ./ svd_h2_obs.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=3)
scatter!( p_S_result, svd_h2_result.S ./ svd_h2_result.S[1], markercolor=:seagreen, ms=3.5, markershape=:circle,msw=1, msc=:seagreen)

savefig("../figure/fig3/yeast_svd_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_svd_resim.pdf"

In [72]:
# mutual information
P_xy = h2_result.weights/sum(h2_result.weights)
P_y = sum(P_xy, dims=1)
P_x = sum(P_xy, dims=2)
is_nan = .!(isnan.(P_xy .* log.(P_xy./(P_x * P_y))))
mi = P_xy .* log.(P_xy./(P_x * P_y))
sum(mi[is_nan])

0.03746276529026015

### histogram

In [70]:
p1 = plot(;xlim=(20,55),  xtick=collect(20:10:55),ylim=(-0.002,0.16),ytick=[0,0.05,0.10,0.15])
stephist!( p1, vcat( [areaobs__[i][isdiv_[i]] for i in eachindex(areaobs__) ]...), bins=collect(20:2:50), normalize=:pdf, lw=0.2#, xlabel="Cell length", ylabel="PDF"
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)

stephist!( p1, vcat( [Lresult[i][isdiv_result[i]] for i in eachindex(Lresult) ]...), normalize=:pdf, bins=collect(20:2:50), lc=:seagreen, lw=0.2)
savefig("../figure/fig3/yeast_histogram_divsize.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_histogram_divsize.pdf"

In [71]:
p2 = plot()
stephist!( p2, vcat( diff.([tobs__[i][isdiv_[i]] for i in eachindex(tobs__)]) ... ), normalize=:pdf,
        xlim=(1,3.5), xtick=collect(0:1:3),ylim=(-0.04,2),ytick=[0,1,2], bins=collect(0:0.2:4), lw=0.2
        , bottommargin=-6Plots.mm, leftmargin=0Plots.mm, background_color=:transparent)
        #,ylabel="PDF", xlabel="Generation time" )
stephist!( p2, vcat( diff.([tresult[i][isdiv_result[i]] for i in eachindex(tresult)]) ... ), normalize=:pdf
, lc=:seagreen, bins=collect(0:0.2:4), lw=0.2)

savefig("../figure/fig3/yeast_histogram_gentime.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_histogram_gentime.pdf"

### nonlinear growth

In [73]:
function interpolation(t, L, deg)
    A = ones(length(L), deg);
    for i = 2:deg
        A[:,i] = t.^(i-1);
    end
    coeff = A\L;
    D = zeros(length(L), deg);
    for i = 2:deg
        D[:,i] = (i-1) * t.^(i-2);
    end
    return coeff, D * coeff, A * coeff
end

interpolation (generic function with 1 method)

In [75]:
bf_g_nonlinear = Polynomial()
Lt_g_nonlinear = []
Diff_g_nonlinear = []
for Li in eachindex(areaobs__)
    pos = findall(isdiv_[Li]);
    for pi = 1:length(pos)-1
        _,diff, Lt = interpolation(tobs__[Li][pos[pi]+1:pos[pi+1]], areaobs__[Li][pos[pi]+1:pos[pi+1]], 5);
        push!(Lt_g_nonlinear, Lt);
        push!(Diff_g_nonlinear, diff);
    end
end
Lt_g_nonlinear = convert(Vector{Vector{Float64}}, Lt_g_nonlinear);
Diff_g_nonlinear = convert(Vector{Vector{Float64}}, Diff_g_nonlinear);
Lt_g_nonlinear = vcat( [ Lt_g_nonlinear[i]  for i in eachindex(Lt_g_nonlinear)]... );
Diff_g_nonlinear = vcat( [Diff_g_nonlinear[i] for i in eachindex(Diff_g_nonlinear) ]... );
Theta_g_nonlinear = gen_bases( Lt_g_nonlinear,  2);
coeffs_g_nonlinear = Theta_g_nonlinear \ Diff_g_nonlinear

3-element Vector{Float64}:
 -8.755160466524018
  1.3928095793927464
 -0.027931909628154113

In [78]:
histogram2d(Lt_g_nonlinear, Diff_g_nonlinear, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-10Plots.mm,
xlim = (5,45), ylims = (-25,45), xtick =[10, 25, 40], ytick=[-20, 0, 20, 40], bins=(5:2.5:45, -25:5:45), c=cgrad(:tempo, rev=false), clim=(0,0.01)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,
plot!(collect(5:0.1:46), coeffs_g_nonlinear[1] .+ coeffs_g_nonlinear[2]*collect(5:0.1:46) .+ coeffs_g_nonlinear[3]*collect(5:0.1:46).^2, lw=1.0, lc=:black, ls=:dash)
savefig("../figure/fig3/yeast_nonlinear_g.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_nonlinear_g.png"

In [46]:
# resimulate data and save or read in your resimulated data
# resimulate data based on your inferred parameters
Lresult = []
tresult = []
for i in eachindex(areaobs__)
    vrj = VariableRateJump(rate_memory, affect!)
    oprob = ODEProblem(f, [areaobs__[i][1];2 * areaobs__[i][1]], (0.0,maximum(tobs__[i])), (bf_g_nonlinear, coeffs_g_nonlinear, bf_h, coeffs_h, Σ1, history_coeffs1[:, imax1], bf1, l01))
    jprob = JumpProblem(oprob, Direct(), vrj)
    sol = solve(jprob,Tsit5(),dtmax=0.01);
    L_r = Array(sol')[:,1];
    t_r = sol.t;
    push!(Lresult, L_r);
    push!(tresult, t_r);
end

Lresult = convert(Vector{Vector{Float64}}, Lresult);
tresult = convert(Vector{Vector{Float64}}, tresult);

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

writedlm("../figure/fig3/yeast_resim_data_nonlinear.txt", [Lresult, tresult], '\t')

In [79]:
# read in resimulated data

using JSON
resim_data = readdlm("../figure/fig3/yeast_resim_data_nonlinear.txt", '\t')

Lresult = [Float64.(JSON.parse(resim_data[1, i])) for i in 1:size(resim_data,2)];
tresult = [Float64.(JSON.parse(resim_data[2, i])) for i in 1:size(resim_data,2)];

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

In [81]:
p_traj = plot(; size=(750,300),xlim=(0,50), ylim=(0,50), xtick=collect(0:25:100), ytick=collect(0:25:50), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent)
linecolor_ = palette(:default, length(area_) )
for i =1:50
    plot!(p_traj, tresult[i], Lresult[i], lc=:snow2, lw=0.2)
end
plot!(p_traj, tresult[3], Lresult[3], lc=:seagreen,lw=0.4)
scatter!(p_traj, tresult[3][isdiv_result[3]], Lresult[3][isdiv_result[3]], markercolor=:seagreen, ms=1,markerstrokewidth=0.2)
p_traj
savefig("../figure/fig3/traj_yeast_nonlinear_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/traj_yeast_nonlinear_resim.pdf"

In [82]:
using StatsBase
hedges = collect(25:1:45);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h2_result = fit(Histogram, (vcat( [Lresult[i][isdiv_result[i]][1:end-1] for i in eachindex(Lresult) ]...), vcat( [Lresult[i][isdiv_result[i]][2:end] for i in eachindex(Lresult) ]...) ), (hedges,hedges));

h2_result_normalize = h2_result.weights / sum(h2_result.weights) / dhedges^2

svd_h2_result = svd(h2_result_normalize);

ll=@layout [a b]
p_h2_result = plot(; aspect_ratio=1, framestyle=:box, xlim=(25,45), ylim=(25,45), xtick=collect(15:10:55), ytick=collect(15:10:55))#, title="P(i,i-1)" 
heatmap!( p_h2_result, hbins, hbins, h2_result_normalize,c = :Blues_9, clim=(0,2e-2),
colorbar_ticks=()) 

plot(p_h2_result, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/fig3/yeast_correlation_nonlinear_resim.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_correlation_nonlinear_resim.png"

In [83]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h2_obs.S ./ svd_h2_obs.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=3)
scatter!( p_S_result, svd_h2_result.S ./ svd_h2_result.S[1], markercolor=:seagreen, ms=3.5, markershape=:circle,msw=1, msc=:seagreen)

savefig("../figure/fig3/yeast_nonlinear_svd_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig3/yeast_nonlinear_svd_resim.pdf"